In [7]:
import os
import json
import shutil
import random

import matplotlib.pyplot as plt


# ==================================================
# PATHS
# ==================================================

SCRIPT_DIR = os.getcwd()

BASE_DIR = os.path.abspath(
    os.path.join(
        SCRIPT_DIR,
        "..",
        "search-dataset",
        "visual_search_dataset_counterfactual",
    )
)


# ==================================================
# IMAGE SETTINGS
# ==================================================

IMAGE_SIZE = 448
DPI = 112
FIGSIZE = (IMAGE_SIZE / DPI, IMAGE_SIZE / DPI)
FONT_SIZE = 24

DISTRACTOR_COUNTS = [
    0, 5, 10, 15, 20,
    25, 30, 35, 40, 45, 50,
]

CONDITIONS = [
    "disjunctive",
    "conjunctive",
]

IMAGES_PER_BIN = 100
NUM_TARGET_PRESENT = IMAGES_PER_BIN // 2

SEED = 42
random.seed(SEED)


# ==================================================
# CANONICAL TARGET POSITIONS
# ==================================================

def generate_canonical_target_positions():
    positions = []

    while len(positions) < NUM_TARGET_PRESENT:
        pos = (
            random.uniform(1.0, 9.0),
            random.uniform(1.0, 9.0),
        )

        if all(
            (pos[0] - existing[0]) ** 2
            + (pos[1] - existing[1]) ** 2
            > 0.65
            for existing in positions
        ):
            positions.append(pos)

    return positions


# ==================================================
# POSITION SAMPLING
# ==================================================

def sample_distractor_positions(
    num_distractors,
    occupied_positions=None,
):
    """
    Sample one spatial layout. Reuse its output for the
    disjunctive and conjunctive counterpart images.
    """
    occupied = list(occupied_positions or [])
    distractor_positions = []

    threshold = 1.0 if num_distractors <= 25 else 0.8

    for _ in range(num_distractors):
        for _attempt in range(1000):
            pos = (
                random.uniform(1.0, 9.0),
                random.uniform(1.0, 9.0),
            )

            if all(
                (pos[0] - existing[0]) ** 2
                + (pos[1] - existing[1]) ** 2
                > threshold
                for existing in occupied
            ):
                distractor_positions.append(pos)
                occupied.append(pos)
                break
        else:
            # Fallback if the layout becomes very crowded.
            pos = (
                random.uniform(1.0, 9.0),
                random.uniform(1.0, 9.0),
            )
            distractor_positions.append(pos)
            occupied.append(pos)

    return distractor_positions


# ==================================================
# SINGLE TRIAL
# ==================================================

def generate_search_trial(
    condition,
    target_present,
    num_distractors,
    save_path,
    distractor_positions,
    target_position=None,
    pair_trial_id=None,
):
    fig, ax = plt.subplots(figsize=FIGSIZE, dpi=DPI)

    fig.subplots_adjust(
        left=0,
        right=1,
        bottom=0,
        top=1,
    )

    ax.set_xlim(0.5, 9.5)
    ax.set_ylim(0.5, 9.5)
    ax.set_aspect("equal")
    ax.axis("off")

    metadata = {
        "condition": condition,
        "target_present": target_present,
        "num_distractors": num_distractors,
        "image_size": IMAGE_SIZE,
        "dpi": DPI,
        "font_size": FONT_SIZE,
        "pair_trial_id": pair_trial_id,
        "objects": [],
    }

    def add_letter(pos, shape, color, role):
        txt = ax.text(
            pos[0],
            pos[1],
            shape,
            color=color,
            fontsize=FONT_SIZE,
            ha="center",
            va="center",
            fontweight="bold",
            fontfamily="DejaVu Sans Mono",
        )

        fig.canvas.draw()
        renderer = fig.canvas.get_renderer()
        bbox = txt.get_window_extent(renderer)

        metadata["objects"].append(
            {
                "role": role,
                "shape": shape,
                "color": color,
                "position_plot": [
                    float(pos[0]),
                    float(pos[1]),
                ],
                "position_pixel": [
                    float(bbox.x0 + bbox.width / 2),
                    float(bbox.y0 + bbox.height / 2),
                ],
                "bbox_pixel": {
                    "x0": float(bbox.x0),
                    "y0": float(bbox.y0),
                    "x1": float(bbox.x1),
                    "y1": float(bbox.y1),
                    "width": float(bbox.width),
                    "height": float(bbox.height),
                },
            }
        )

    # Target: identical location in both paired conditions.
    if target_present:
        add_letter(
            target_position,
            shape="L",
            color="green",
            role="target",
        )

    # Distractors: identical LOCATIONS in both paired conditions.
    for pos in distractor_positions:
        if condition == "disjunctive":
            shape = "L"
            color = "red"
        elif condition == "conjunctive":
            shape, color = random.choice(
                [
                    ("L", "red"),
                    ("T", "green"),
                ]
            )
        else:
            raise ValueError(f"Unknown condition: {condition}")

        add_letter(
            pos,
            shape=shape,
            color=color,
            role="distractor",
        )

    fig.savefig(save_path, dpi=DPI)
    plt.close(fig)

    json_path = save_path.replace(".png", ".json")
    with open(json_path, "w") as f:
        json.dump(metadata, f, indent=2)


# ==================================================
# DATASET GENERATION
# ==================================================

def generate_dataset():
    if os.path.exists(BASE_DIR):
        print("Deleting old dataset...")
        shutil.rmtree(BASE_DIR)

    for condition in CONDITIONS:
        os.makedirs(
            os.path.join(BASE_DIR, condition),
            exist_ok=True,
        )

    canonical_target_positions = generate_canonical_target_positions()

    for num_distractors in DISTRACTOR_COUNTS:
        print(f"\nGenerating matched pairs: distractors={num_distractors}")

        # ----------------------------------------------
        # Target-present pairs
        # ----------------------------------------------
        for trial in range(NUM_TARGET_PRESENT):
            target_position = canonical_target_positions[trial]

            # Sample positions ONCE; reuse for both conditions.
            distractor_positions = sample_distractor_positions(
                num_distractors=num_distractors,
                occupied_positions=[target_position],
            )

            pair_trial_id = (
                f"distractors_{num_distractors}_"
                f"trial_{trial:03d}_target_true"
            )

            for condition in CONDITIONS:
                filename = (
                    f"{condition}_distractors_{num_distractors}"
                    f"_trial_{trial:03d}_target_true.png"
                )

                generate_search_trial(
                    condition=condition,
                    target_present=True,
                    num_distractors=num_distractors,
                    save_path=os.path.join(
                        BASE_DIR,
                        condition,
                        filename,
                    ),
                    target_position=target_position,
                    distractor_positions=distractor_positions,
                    pair_trial_id=pair_trial_id,
                )

        # ----------------------------------------------
        # Target-absent pairs
        # ----------------------------------------------
        for trial in range(NUM_TARGET_PRESENT, IMAGES_PER_BIN):
            # Again, sample positions ONCE and reuse them.
            distractor_positions = sample_distractor_positions(
                num_distractors=num_distractors,
                occupied_positions=[],
            )

            pair_trial_id = (
                f"distractors_{num_distractors}_"
                f"trial_{trial:03d}_target_false"
            )

            for condition in CONDITIONS:
                filename = (
                    f"{condition}_distractors_{num_distractors}"
                    f"_trial_{trial:03d}_target_false.png"
                )

                generate_search_trial(
                    condition=condition,
                    target_present=False,
                    num_distractors=num_distractors,
                    save_path=os.path.join(
                        BASE_DIR,
                        condition,
                        filename,
                    ),
                    target_position=None,
                    distractor_positions=distractor_positions,
                    pair_trial_id=pair_trial_id,
                )

        print(f"  distractors={num_distractors} done")

    print("\nDone.")


# ==================================================
# MAIN
# ==================================================

if __name__ == "__main__":
    generate_dataset()

Deleting old dataset...

Generating disjunctive
  distractors=0 done
  distractors=5 done
  distractors=10 done
  distractors=15 done
  distractors=20 done
  distractors=25 done
  distractors=30 done
  distractors=35 done
  distractors=40 done
  distractors=45 done
  distractors=50 done
Finished disjunctive

Generating conjunctive
  distractors=0 done
  distractors=5 done
  distractors=10 done
  distractors=15 done
  distractors=20 done
  distractors=25 done
  distractors=30 done
  distractors=35 done
  distractors=40 done
  distractors=45 done
  distractors=50 done
Finished conjunctive

Done.


# Disjunctive T

In [1]:
import glob
import json
import os
import matplotlib.pyplot as plt

# ==================================================
# PATHS
# ==================================================

BASE_DIR = "/oscar/data/epavlick/aparasel/Binding-Problem-Brown/search-dataset/visual_search_dataset_counterfactual"
CONJUNCTIVE_DIR = os.path.join(BASE_DIR, "conjunctive")
OUTPUT_DIR = os.path.join(BASE_DIR, "disjunctive_T")

# ==================================================
# RENDERING & RECONSTRUCTION
# ==================================================

def render_trial_from_metadata(metadata, save_path):
    """
    Renders and saves a trial image using the exact coordinates and dimensions
    stored in the metadata, and updates bounding box metadata.
    """
    image_size = metadata.get("image_size", 448)
    dpi = metadata.get("dpi", 112)
    font_size = metadata.get("font_size", 24)
    figsize = (image_size / dpi, image_size / dpi)

    fig, ax = plt.subplots(figsize=figsize, dpi=dpi)
    fig.subplots_adjust(left=0, right=1, bottom=0, top=1)

    ax.set_xlim(0.5, 9.5)
    ax.set_ylim(0.5, 9.5)
    ax.set_aspect("equal")
    ax.axis("off")

    updated_objects = []

    for obj in metadata["objects"]:
        pos = obj["position_plot"]
        shape = obj["shape"]
        color = obj["color"]
        role = obj["role"]

        txt = ax.text(
            pos[0],
            pos[1],
            shape,
            color=color,
            fontsize=font_size,
            ha="center",
            va="center",
            fontweight="bold",
            fontfamily="DejaVu Sans Mono",
        )

        fig.canvas.draw()
        renderer = fig.canvas.get_renderer()
        bbox = txt.get_window_extent(renderer)

        updated_objects.append({
            "role": role,
            "shape": shape,
            "color": color,
            "position_plot": [float(pos[0]), float(pos[1])],
            "position_pixel": [
                float(bbox.x0 + bbox.width / 2),
                float(bbox.y0 + bbox.height / 2),
            ],
            "bbox_pixel": {
                "x0": float(bbox.x0),
                "y0": float(bbox.y0),
                "x1": float(bbox.x1),
                "y1": float(bbox.y1),
                "width": float(bbox.width),
                "height": float(bbox.height),
            },
        })

    # Save reconstructed image
    fig.savefig(save_path, dpi=dpi)
    plt.close(fig)

    # Return updated metadata dictionary
    new_metadata = dict(metadata)
    new_metadata["condition"] = "disjunctive_T"
    new_metadata["objects"] = updated_objects
    return new_metadata


def generate_disjunctive_t_counterfactuals():
    if not os.path.exists(CONJUNCTIVE_DIR):
        raise FileNotFoundError(f"Source folder not found: {CONJUNCTIVE_DIR}")

    os.makedirs(OUTPUT_DIR, exist_ok=True)

    json_files = sorted(glob.glob(os.path.join(CONJUNCTIVE_DIR, "*.json")))
    print(f"Found {len(json_files)} conjunctive trials to convert...")

    for idx, src_json_path in enumerate(json_files):
        with open(src_json_path, "r") as f:
            meta = json.load(f)

        # Mutate distractors to Green Ts while preserving roles and target
        for obj in meta["objects"]:
            if obj["role"] == "distractor":
                obj["shape"] = "T"
                obj["color"] = "green"

        # Generate target filenames (e.g. replace prefix 'conjunctive_' with 'disjunctive_T_')
        src_fname = os.path.basename(src_json_path)
        base_name = src_fname.replace("conjunctive_", "disjunctive_T_").replace(".json", "")
        
        dst_png_path = os.path.join(OUTPUT_DIR, f"{base_name}.png")
        dst_json_path = os.path.join(OUTPUT_DIR, f"{base_name}.json")

        # Re-render with new distractor features
        updated_meta = render_trial_from_metadata(meta, dst_png_path)

        with open(dst_json_path, "w") as f:
            json.dump(updated_meta, f, indent=2)

        if (idx + 1) % 100 == 0 or (idx + 1) == len(json_files):
            print(f"Processed {idx + 1}/{len(json_files)} files.")

    print(f"\nDone! All files successfully generated in: {OUTPUT_DIR}")


if __name__ == "__main__":
    generate_disjunctive_t_counterfactuals()

Found 1100 conjunctive trials to convert...
Processed 100/1100 files.
Processed 200/1100 files.
Processed 300/1100 files.
Processed 400/1100 files.
Processed 500/1100 files.
Processed 600/1100 files.
Processed 700/1100 files.
Processed 800/1100 files.
Processed 900/1100 files.
Processed 1000/1100 files.
Processed 1100/1100 files.

Done! All files successfully generated in: /oscar/data/epavlick/aparasel/Binding-Problem-Brown/search-dataset/visual_search_dataset_counterfactual/disjunctive_T


In [8]:
from PIL import Image
from collections import Counter
import os

sizes = []

for root, _, files in os.walk(BASE_DIR):
    for f in files:
        if f.endswith(".png"):
            sizes.append(
                Image.open(
                    os.path.join(root, f)
                ).size
            )

print(Counter(sizes))

Counter({(448, 448): 2200})


# Green Mask source and T Mask source

In [1]:
import copy
import glob
import json
import os
import matplotlib.pyplot as plt

# ==================================================
# PATHS
# ==================================================

BASE_DIR = (
    "/oscar/data/epavlick/aparasel/Binding-Problem-Brown/"
    "search-dataset/visual_search_dataset_counterfactual"
)

CONJUNCTIVE_DIR = os.path.join(BASE_DIR, "conjunctive")

OUTPUT_DIRS = {
    "src_green_mask": os.path.join(BASE_DIR, "src_green_mask"),
    "src_L_mask": os.path.join(BASE_DIR, "src_L_mask"),
}

# ==================================================
# RENDERING & RECONSTRUCTION
# ==================================================

def render_trial_from_metadata(metadata, save_path, new_condition_name):
    """
    Renders and saves a trial image using the exact coordinates stored in
    metadata, and recalculates accurate pixel bounding boxes for the new glyphs.
    """
    image_size = metadata.get("image_size", 448)
    dpi = metadata.get("dpi", 112)
    font_size = metadata.get("font_size", 24)
    figsize = (image_size / dpi, image_size / dpi)

    fig, ax = plt.subplots(figsize=figsize, dpi=dpi)
    fig.subplots_adjust(left=0, right=1, bottom=0, top=1)

    ax.set_xlim(0.5, 9.5)
    ax.set_ylim(0.5, 9.5)
    ax.set_aspect("equal")
    ax.axis("off")

    updated_objects = []

    for obj in metadata["objects"]:
        pos = obj["position_plot"]
        shape = obj["shape"]
        color = obj["color"]
        role = obj["role"]

        txt = ax.text(
            pos[0],
            pos[1],
            shape,
            color=color,
            fontsize=font_size,
            ha="center",
            va="center",
            fontweight="bold",
            fontfamily="DejaVu Sans Mono",
        )

        fig.canvas.draw()
        renderer = fig.canvas.get_renderer()
        bbox = txt.get_window_extent(renderer)

        updated_objects.append({
            "role": role,
            "shape": shape,
            "color": color,
            "position_plot": [float(pos[0]), float(pos[1])],
            "position_pixel": [
                float(bbox.x0 + bbox.width / 2),
                float(bbox.y0 + bbox.height / 2),
            ],
            "bbox_pixel": {
                "x0": float(bbox.x0),
                "y0": float(bbox.y0),
                "x1": float(bbox.x1),
                "y1": float(bbox.y1),
                "width": float(bbox.width),
                "height": float(bbox.height),
            },
        })

    # Save reconstructed image
    fig.savefig(save_path, dpi=dpi)
    plt.close(fig)

    # Return deep-copied new metadata
    new_meta = copy.deepcopy(metadata)
    new_meta["condition"] = new_condition_name
    new_meta["objects"] = updated_objects
    return new_meta


# ==================================================
# GENERATE MASKED DATASETS
# ==================================================

def generate_masked_datasets():
    if not os.path.exists(CONJUNCTIVE_DIR):
        raise FileNotFoundError(f"Source folder not found: {CONJUNCTIVE_DIR}")

    # Create destination directories
    for folder_path in OUTPUT_DIRS.values():
        os.makedirs(folder_path, exist_ok=True)

    json_files = sorted(glob.glob(os.path.join(CONJUNCTIVE_DIR, "*.json")))
    print(f"Found {len(json_files)} source conjunctive trials to process.")

    # Define the transformations
    masks_config = {
        "src_green_mask": {
            "target": {"shape": "T", "color": "green"},     # Target becomes Green T
            "distractor": {"shape": "T", "color": "red"},   # Distractors become Red Ts
        },
        "src_L_mask": {
            "target": {"shape": "L", "color": "red"},       # Target becomes Red L
            "distractor": {"shape": "T", "color": "red"},   # Distractors become Red Ts
        },
    }

    for mask_name, config in masks_config.items():
        dest_dir = OUTPUT_DIRS[mask_name]
        print(f"\nGenerating {mask_name} in {dest_dir}...")

        for idx, src_json_path in enumerate(json_files):
            # Read-only load of existing metadata
            with open(src_json_path, "r") as f:
                meta = json.load(f)

            # Deep copy to ensure original objects remain unmutated
            trial_meta = copy.deepcopy(meta)

            for obj in trial_meta["objects"]:
                if obj["role"] == "target":
                    obj["shape"] = config["target"]["shape"]
                    obj["color"] = config["target"]["color"]
                elif obj["role"] == "distractor":
                    obj["shape"] = config["distractor"]["shape"]
                    obj["color"] = config["distractor"]["color"]

            # Construct output filenames
            src_fname = os.path.basename(src_json_path)
            base_name = src_fname.replace("conjunctive_", f"{mask_name}_").replace(".json", "")

            dst_png_path = os.path.join(dest_dir, f"{base_name}.png")
            dst_json_path = os.path.join(dest_dir, f"{base_name}.json")

            # Render image and save new metadata
            updated_meta = render_trial_from_metadata(
                metadata=trial_meta,
                save_path=dst_png_path,
                new_condition_name=mask_name,
            )

            with open(dst_json_path, "w") as f:
                json.dump(updated_meta, f, indent=2)

            if (idx + 1) % 100 == 0 or (idx + 1) == len(json_files):
                print(f"  [{mask_name}] Processed {idx + 1}/{len(json_files)} trials")

    print("\nAll counterfactual masked datasets successfully generated!")


if __name__ == "__main__":
    generate_masked_datasets()

Found 1100 source conjunctive trials to process.

Generating src_green_mask in /oscar/data/epavlick/aparasel/Binding-Problem-Brown/search-dataset/visual_search_dataset_counterfactual/src_green_mask...
  [src_green_mask] Processed 100/1100 trials
  [src_green_mask] Processed 200/1100 trials
  [src_green_mask] Processed 300/1100 trials
  [src_green_mask] Processed 400/1100 trials
  [src_green_mask] Processed 500/1100 trials
  [src_green_mask] Processed 600/1100 trials
  [src_green_mask] Processed 700/1100 trials
  [src_green_mask] Processed 800/1100 trials
  [src_green_mask] Processed 900/1100 trials
  [src_green_mask] Processed 1000/1100 trials
  [src_green_mask] Processed 1100/1100 trials

Generating src_L_mask in /oscar/data/epavlick/aparasel/Binding-Problem-Brown/search-dataset/visual_search_dataset_counterfactual/src_L_mask...
  [src_L_mask] Processed 100/1100 trials
  [src_L_mask] Processed 200/1100 trials
  [src_L_mask] Processed 300/1100 trials
  [src_L_mask] Processed 400/1100 t